In [1]:
import os
import cassiopeia as cas
import pandas as pd
import numpy as np
import networkx as nx
import pickle
from ete3 import Tree


In [2]:

# read in an allele table
allele_table = pd.read_csv("https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4905nnn/GSM4905334/suppl/GSM4905334_alleleTable.5k.txt.gz", sep='\t',
                           usecols = ['cellBC', 'intBC', 'r1', 'r2', 'r3', 'allele', 'LineageGroup', 'sampleID', 'readCount', 'UMI'])
allele_table.head(5)


In [3]:
allele_table[allele_table['LineageGroup']==81]

In [4]:
indel_priors = cas.pp.compute_empirical_indel_priors(allele_table, grouping_variables=['intBC', 'LineageGroup'])
indel_priors.sort_values(by='count', ascending=False).head()


### Make a tree for each clone, save this data for Metient input

In [5]:
TISSUE_ORDER = ['LL',"RE","RW","M1","M2","Liv"]

def write_tree_to_txt(clone, nx_tree, output_dir):
    adj_matrix = nx.to_numpy_array(nx_tree, dtype=int)
    
    rows, cols = adj_matrix.shape
    with open(os.path.join(output_dir, f"{clone}_tree.txt"), "w") as f:
        for i in range(rows):
            for j in range(cols):
                if adj_matrix[i, j] == 1:
                    f.write(f"{i} {j}\n")

def shorten_node_str(s):
    if "cassiopeia_internal_node" in s:
        return "IN"
    else:
        return s[:7]
    
def write_tsv(clone, cell_info, nx_tree, output_dir):
    data = []
    anat_sites = list(set(["LL"]+list(cell_info['sampleID'].unique())))
    print(anat_sites)
    for node_index, node in enumerate(nx_tree.nodes()):
        detected_site = None
        if node in cell_info.index:
            detected_site = cell_info.loc[[node]]['sampleID'].item()
        for site_idx,site in enumerate(anat_sites):
            in_site = int(detected_site == site)
            site_category = 'primary' if site == 'LL' else 'metastasis'
            data.append([site_idx,site,node_index,node,in_site,site_category,1])
    df = pd.DataFrame(data,columns=['anatomical_site_index','anatomical_site_label','cluster_index',
                                    'cluster_label','present','site_category','num_mutations'])
    df.to_csv(os.path.join(output_dir, f"{clone}.tsv"),index=False, sep="\t")
    return df

from itertools import product
def generate_all_histories(node_to_possible_labels):
    """
    Generate all possible migration histories by permuting labels for each node.
    
    Args:
        graph (nx.DiGraph): A directed graph representing the migration paths.
        labels (dict): A dictionary mapping nodes to lists of possible labels.
    
    Returns:
        list of dict: A list of dictionaries representing all possible migration histories.
    """
    # Prepare a list of nodes in the graph
    nodes = list(node_to_possible_labels.keys())
    
    # Create a list of label combinations for each node
    label_combinations = []
    for node in nodes:
        label_combinations.append(node_to_possible_labels[node])
     

    # Compute the Cartesian product of the label combinations
    all_label_combinations = product(*label_combinations)
    print("possible combinations:", len(all_label_combinations))
    # Create histories from the combinations
    all_histories = []
    for combination in all_label_combinations:
        history = {}
        for node, label in zip(nodes, combination):
            history[node] = label
        all_histories.append(history)

    return all_histories


def calculate_min_migrations(G, root, node_to_label):
    """
    Calculate the parsimony score based on the number of label changes in the current history.
    """
    dfs_edges = nx.dfs_edges(G, source=root)
    num_migrations = 0
    for (parent, child) in dfs_edges:
        if node_to_label[parent] != node_to_label[child]:
            num_migrations += 1
    return num_migrations
       

def find_all_min_mig_histories(G, root, all_labelings, min_mig_score):
    min_mig_histories = []
    for labeling in all_labelings:
        num_migrations = calculate_min_migrations(G, root, labeling)
        if num_migrations == min_mig_score:
            min_mig_histories.append(labeling)
    return min_mig_histories

def count_label_combinations(label_combinations):
    """
    Count the number of possible label combinations from label_combinations.
    
    Args:
        label_combinations (list of lists): A list where each element is a list of possible labels for a node.
        
    Returns:
        int: The total number of possible combinations.
    """
    # Calculate the product of the lengths of each label list
    total_combinations = 1
    for labels in label_combinations:
        total_combinations *= len(labels)
    
    return total_combinations


In [6]:

lt_dir_prefix = "/data/morrisq/divyak/projects/metient/metient/data/quinn_lt_2021/"
lt_cm_dir = "/data/morrisq/divyak/data/quinn_lin_tracing_2021/GSE161363/character_matrices"
metient_input_dir = os.path.join(lt_dir_prefix,"metient_inputs/")
# For outputs from cassiopeia inference
cass_dir = os.path.join(lt_dir_prefix, "cassiopeia_outputs")

meta_df = pd.read_csv("/data/morrisq/divyak/data/quinn_lin_tracing_2021/GSE161363/GSM4905335_meta.5k.tsv", sep='\t', index_col=0)
clone_to_random_min_mig_labeling, clone_to_tissue_trans, clone_to_min_mig_score = {},{},{}

for clone in allele_table['LineageGroup'].unique():    
    clone_allele_table = allele_table[allele_table['LineageGroup'] == clone]

    n_cells = clone_allele_table['cellBC'].nunique()
    n_intbc = clone_allele_table['intBC'].nunique()

    
    character_matrix, priors, state_2_indel = cas.pp.convert_alleletable_to_character_matrix(clone_allele_table,
                                                                                             allele_rep_thresh = 0.9,
                                                                                             mutation_priors = indel_priors)
    cas_tree = cas.data.CassiopeiaTree(character_matrix=character_matrix, priors=priors)
    # For some reason, the character matrix
    if clone == 81:
        character_matrix = pd.read_csv(os.path.join(lt_cm_dir, f"m5k_lg{clone}_character_matrix.alleleThresh.txt"), index_col=0,sep="\t")
        cas_tree = cas.data.CassiopeiaTree(character_matrix=character_matrix)

    cell_meta = clone_allele_table.groupby('cellBC').agg({"intBC": 'nunique', 'UMI': 'sum', 'sampleID': 'unique'})
    cell_meta['sampleID'] = [x[0] for x in cell_meta['sampleID']]

    missing_proportion = (character_matrix == -1).sum(axis=0) / character_matrix.shape[0]
    uncut_proportion = (character_matrix == 0).sum(axis=0) / character_matrix.shape[0]
    n_unique_states = character_matrix.apply(lambda x: len(np.unique(x[(x != 0) & (x != -1)])), axis=0)
    print(f"\nClonal population #{clone} has {n_cells} cells and {n_intbc} intBCs ({n_intbc * 3}) characters.")

    character_meta = pd.DataFrame([missing_proportion, uncut_proportion, n_unique_states], index = ['missing_prop', 'uncut_prop', 'n_unique_states']).T
    cas_tree.cell_meta = cell_meta
    cas_tree.character_meta = character_meta
    vanilla_greedy = cas.solver.VanillaGreedySolver()
    
    ilp_solver = cas.solver.ILPSolver(convergence_time_limit=12600,
                                      maximum_potential_graph_layer_size=10000,
                                      seed=1234)
    hybrid_solver = cas.solver.HybridSolver(top_solver=vanilla_greedy, bottom_solver=ilp_solver, cell_cutoff=40, threads=8)

    # reconstruct the tree
    try:
        vanilla_greedy.solve(cas_tree, collapse_mutationless_edges=True)
        #hybrid_solver.solve(cas_tree)
    except Exception as e:
        print('Couldnt run tree solver on clone', clone, e, "\n")
        continue
    
    G = cas_tree._CassiopeiaTree__network
    nodes = G.nodes

    # Use Fitch-Hartigan algorithm
    cas.tools.fitch_hartigan(cas_tree, 'sampleID', state_key="state", label_key="label")
    random_min_mig_labeling = {}
    node_to_min_mig_labels = {}
    for i,node in enumerate(G.nodes()):
        node_to_min_mig_labels[node] = nodes[node]['state']
        random_min_mig_labeling[node] = nodes[node]['label']

    clone_to_random_min_mig_labeling[clone] = random_min_mig_labeling
    
    min_mig_score = cas.tools.score_small_parsimony(cas_tree, 'sampleID', label_key="label")
    clone_to_min_mig_score[clone] = min_mig_score
    print("min_mig_score", min_mig_score)

    count_arr = cas.tools.fitch_count(cas_tree, 'sampleID')
    np.fill_diagonal(count_arr.values,0)
    count_arr = count_arr.apply(lambda x: x / max(1, x.sum()), axis=1)
    clone_to_tissue_trans[clone] = count_arr

    # cas.pl.plot_matplotlib(cas_tree)
    # num_possible_histories = count_label_combinations(node_to_min_mig_labels)
    # print("num possible histories","{:.2e}".format(Decimal(num_possible_histories)))
    # if num_possible_histories < 100:
    #     all_histories = generate_all_histories(node_to_min_mig_labels)
    #     print("all_histories", len(all_histories))
    #     min_mig_histories = find_all_min_mig_histories(G, cas_tree.root, all_histories, min_mig_score)
    
    # Find the root node (the one without parents/in-degree 0)
    root_node = [node for node, degree in G.in_degree() if degree == 0][0]
    # Add a new "primary" node
    G.add_node("primary")
    G.add_edge("primary", root_node)
    write_tree_to_txt(clone, G, metient_input_dir)

    write_tsv(clone, cas_tree.cell_meta, G, metient_input_dir)
    with open(os.path.join(cass_dir, f'{clone}_vanilla_cas_networkx.pkl'), 'wb') as file:
        pickle.dump(G, file)
   

In [7]:

np_data = {key: (df.to_numpy(), list(df.index)) for key, df in clone_to_tissue_trans.items()}

with open(os.path.join(cass_dir,'vanilla_greedy_cas_tissue_transitions.pkl'), 'wb') as f:
    pickle.dump(np_data, f)

with open(os.path.join(cass_dir,'vanilla_greedy_cas_random_min_mig_history.pkl'), 'wb') as f:
    pickle.dump(clone_to_random_min_mig_labeling, f)

with open(os.path.join(cass_dir,'vanilla_greedy_cas_min_mig_score.pkl'), 'wb') as f:
    pickle.dump(clone_to_min_mig_score, f)